# RoomRunner pickup: train ACT on Colab

Trains a LeRobot ACT policy on the pickup demonstrations recorded with `record.py`.

1. **Runtime > Change runtime type > GPU** (T4 is enough).
2. On the robot PC: `cd ~/roomrunner_data && zip -r roomrunner_pick.zip roomrunner_pick`, then upload
   `roomrunner_pick.zip` to Google Drive in a folder `roomrunner/`.
3. Run the cells in order. Checkpoints are saved to Drive (`roomrunner/act_pick`), so a disconnect loses at
   most `SAVE_FREQ` steps: run the *resume* cell to go on.
4. Download `roomrunner/act_pick_last.zip` and unzip it on the robot PC into `~/roomrunner_data/act_pick`
   (README, step 5).

Same LeRobot version as the robot PC (0.6.2), so the checkpoint loads there.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/roomrunner'

In [ ]:
!pip install -q "lerobot[training]==0.6.2"

In [ ]:
!mkdir -p /content/data && unzip -q -o "$DRIVE/roomrunner_pick.zip" -d /content/data && ls /content/data/roomrunner_pick

In [ ]:
# Quick look: episodes per task, frames, cameras
from lerobot.datasets.lerobot_dataset import LeRobotDataset
ds = LeRobotDataset('team1/roomrunner_pick', root='/content/data/roomrunner_pick')
print(ds.num_episodes, 'episodes,', ds.num_frames, 'frames,', ds.fps, 'fps, cameras', ds.meta.camera_keys)

## Train

`STEPS`: about 30k to 50k steps is a good start for ~60 episodes (a T4 does roughly 5 to 8 steps/s at batch 8,
so 40k steps is about 1.5 to 2 h). Watch the loss in the log; it should fall steadily and flatten.

In [ ]:
STEPS, BATCH, SAVE_FREQ = 40000, 8, 5000
OUT = f'{DRIVE}/act_pick'
!lerobot-train --policy.type=act --dataset.repo_id=team1/roomrunner_pick --dataset.root=/content/data/roomrunner_pick \
  --output_dir="$OUT" --job_name=act_pick --steps=$STEPS --batch_size=$BATCH --save_freq=$SAVE_FREQ \
  --num_workers=2 --policy.device=cuda --policy.push_to_hub=false --wandb.enable=false

## Resume (only after a disconnect)
Run the Drive, pip and unzip cells again first.

In [ ]:
!lerobot-train --resume=true --config_path="$OUT/checkpoints/last/pretrained_model/train_config.json"

## Pack the last checkpoint for the robot PC

In [ ]:
!cd "$OUT" && zip -q -r "$DRIVE/act_pick_last.zip" checkpoints/last && ls -lh "$DRIVE/act_pick_last.zip"